### Control Flow

[Docs](https://docs.python.org/3/tutorial/controlflow.html#)

## Why does control flow exist?

A program is, by default, just a list of instructions executed top to bottom, once. That's enough for a script that does one fixed thing. It is **not** enough for anything that has to:

- **decide** — do A or B depending on data it only sees at runtime → `if / elif / else`
- **repeat** — do the same thing N times, or once per item, or "until some condition holds" → `for`, `while`
- **exit early** — stop repeating the moment the goal is met, without waiting for the "natural" end → `break`
- **skip selectively** — abandon just *this* iteration without abandoning the whole loop → `continue`
- **react to how the loop ended** — tell the difference between "finished normally" and "got cut short" → the loop `else` clause

Each tool below exists because someone hit a specific, recurring shape of problem and the naive solution (extra boolean flags, duplicated exit checks, deeply nested ifs) was error-prone or unreadable. Keep asking *"what would I have to write without this keyword?"* — that's where the motivation lives.

**Roadmap for this notebook:**
1. `if / elif / else` — quick recap (you already know this)
2. `break` / `continue` — stop the loop vs. skip this round
3. The loop `else` clause — the one that trips people up
4. Internals — what's actually happening under the hood
5. Gotchas, decision rules, interview angle
6. Practice problems (below the tutorial)


### 1. `if / elif / else` — quick recap

**Decision rule, not just syntax:** `if` picks *one* branch out of several based on a condition known only at runtime. `elif` chains mutually-exclusive checks (only the first true one runs). `else` is the "none of the above" catch-all — it is optional, and skipping it silently is a common bug source (data that matches nothing just falls through unhandled).

You already use this constantly (validating a request payload, branching on an HTTP status code, checking `if user is None`). The one habit worth tightening: prefer guard clauses (`if not valid: return`) over deep nesting — flatter code reads top-to-bottom like a checklist instead of a pyramid.

In [ ]:
def describe_status(status_code: int) -> str:
    """Branch on an HTTP-style status code — a decision, not a repetition."""
    if 200 <= status_code < 300:
        return "success"
    elif 300 <= status_code < 400:
        return "redirect"
    elif 400 <= status_code < 500:
        return "client error"
    elif 500 <= status_code < 600:
        return "server error"
    else:
        return "unknown"  # the catch-all — without it, weird codes vanish silently


for code in (204, 301, 404, 503, 999):
    print(f"{code}: {describe_status(code)}")


### 2. `break` and `continue` — stop the loop vs. skip this round

**The problem they solve:** a bare `for`/`while` loop always runs its body for every item, start to finish. Two very common needs don't fit that:

- *"I found what I was looking for — stop looping entirely."* Without `break` you'd need a flag variable checked at the top of every remaining iteration (`if not found: ...`), wasting cycles and adding a variable purely to fake an early exit.
- *"This particular item is invalid/irrelevant — skip it, but keep processing the rest."* Without `continue` you'd wrap the rest of the loop body in an `if valid:` block, and every new check nests one level deeper.

**Plain language:**
- `break` = **stop the assembly line.** The whole loop ends immediately, right now, regardless of how many items were left.
- `continue` = **toss this one item off the line and move to the next.** The loop keeps running; only the remaining code *for this iteration* is skipped.

**Analogy:** Imagine reviewing a stack of pull requests. `break` is "I found the PR that broke prod — stop reviewing, go fix it now." `continue` is "this PR is just a typo fix, nothing to review — next one."

**Real-world tie-in:** `continue` is your bread and butter in data-cleaning/ETL loops (skip malformed rows, keep the batch running). `break` shows up in early-exit search loops and "stop once you have enough results" pagination loops.

In [ ]:
# --- break: stop as soon as the goal is met ---
users = [{"id": 1, "role": "viewer"}, {"id": 2, "role": "admin"}, {"id": 3, "role": "viewer"}]

for user in users:
    if user["role"] == "admin":
        print(f"found admin: {user}")
        break  # no point checking the remaining users
else:
    print("no admin found")  # preview of section 3 — ignore for now


# --- continue: skip malformed rows, keep the batch alive ---
raw_rows = ["12", "not-a-number", "7", "", "42"]
total = 0

for row in raw_rows:
    if not row.isdigit():
        continue  # bad row: skip it, do NOT abort the whole import
    total += int(row)

print(f"sum of valid rows: {total}")


**Conditional vs. unconditional loops — a distinction worth naming explicitly:**

- A **conditional loop** (`while some_condition:`, or a `for` over a known collection) has its number of iterations determined by something you can point to *before* the loop runs.
- An **unconditional loop** (`while True:`) has no built-in stopping point at all — by construction it would run forever. The *only* way it ends is an internal decision made mid-body: a `break` (or `return`/`raise`). This is the standard idiom for "keep going until something happens that I can only detect from inside the loop" — reading input until a sentinel value, polling until a resource is ready, retrying until success.

Mixing them is normal and idiomatic: the outer loop is unconditional (`while True`), and a conditional `if` inside it decides *when* to fire the `break`.

In [ ]:
# Unconditional loop (while True) + conditional exit (if/break) — the "sentinel" idiom.
# Simulated here with a queue of canned inputs instead of real input(), so it stays runnable.
simulated_inputs = iter(["help", "status", "quit"])

while True:  # unconditional: no stopping condition of its own
    command = next(simulated_inputs)
    print(f"> {command}")

    if command == "quit":   # conditional: this is what actually ends it
        print("bye!")
        break
    elif command == "help":
        print("commands: help, status, quit")
    else:
        print(f"unknown command: {command}")


**Gotchas:**

- `break` / `continue` only affect the **nearest enclosing loop**. Inside nested loops, `break` escapes the inner loop only — the outer loop keeps going. (There is no `break 2` in Python; see the nested-loop problem below for the workaround.)
- In a `while` loop, `continue` jumps straight back to the **condition check** — it does *not* run whatever code updates your loop variable, if that update was written after the `continue`. Forgetting this is a classic way to write an infinite loop. In a `for` loop this can't happen, because the iterator advances automatically regardless of `continue`.
- `break`/`continue` inside a `try/finally` still trigger the `finally` block before actually breaking/continuing — cleanup code is not skipped.

### 3. The loop `else` clause — the one that trips people up

**The problem it solves:** you loop looking for something, and afterwards you need to know *which way the loop ended*: did it `break` because it found/succeeded, or did it run every iteration and find nothing? The naive fix is a boolean flag: set `found = False` before the loop, set `found = True` right where you'd `break`, then check the flag after the loop. That works, but it's an extra variable that exists purely to answer one yes/no question the loop already "knows" the answer to.

`else` on a `for`/`while` answers that question for you, with no flag:

> **The loop's `else` block runs if and only if the loop finished without hitting a `break`.**

**The single biggest misconception to kill right now:** `else` does **not** mean "if the loop ran" and does **not** mean "if the loop found nothing to iterate over." An empty `for x in []:` still runs its `else` (no break ever happened — trivially true). `else` is entirely about *how the loop exited*, not about how many times it iterated.

**Plain language:** think of it as "the loop's own success handler." `break` is the loop's way of saying "I'm leaving early, on purpose, because I got what I came for." `else` is "I made it to the natural end without that happening."

**Analogy:** you're searching a filing cabinet drawer by drawer for one document (`for drawer in cabinet:`). The moment you find it, you slam the cabinet shut and stop (`break`) — you don't also run the "document not found" procedure, because you obviously don't need to. But if you check every single drawer and never find it, only *then* do you run the "not found, escalate" procedure (`else`). You never wrote down "found = False" on a sticky note first — the fact that you're still standing there having checked every drawer *is* the signal.

**The unifying insight (this is the "aha" that makes it click):** Python's `else` on `try` behaves by the *exact same rule*, just swapping "break" for "exception": `try/except/else` runs its `else` only if the `try` block completed **without raising**. In both cases, `else` means *"the block completed normally, with no early/abnormal exit."* Once you see `for-else`, `while-else`, and `try-else` as one rule instead of three, the confusion mostly disappears.

In [ ]:
users = [{"id": 1, "role": "viewer"}, {"id": 2, "role": "viewer"}, {"id": 3, "role": "viewer"}]


def find_admin_with_flag(users):
    """The pattern for-else replaces: an extra variable just to remember 'did we break?'."""
    found_admin = None
    for user in users:
        if user["role"] == "admin":
            found_admin = user
            break
    if found_admin is None:
        raise ValueError("no admin found")
    return found_admin


def find_admin_with_for_else(users):
    """Same logic, zero flag variables — the loop's own exit path tells the story."""
    for user in users:
        if user["role"] == "admin":
            return user
    else:  # ran to completion: no user matched, no break happened
        raise ValueError("no admin found")


for finder in (find_admin_with_flag, find_admin_with_for_else):
    try:
        finder(users)
    except ValueError as exc:
        print(f"{finder.__name__}: {exc}")

# The "misconception killer": else runs even when the loop body never executes at all.
for _ in []:
    print("never printed")
else:
    print("ran with zero iterations — no break happened, so else still fires")


**`while-else` — same rule, applied to a *conditional* loop.** The `else` runs if the `while` condition simply became falsy, and skips if a `break` fired instead. This is the canonical **retry-until-success** shape: a *conditional* loop (`while attempts_left:`) bounds how many tries you get, `break` marks "succeeded early," and `else` marks "ran out of attempts."

**A deliberate trap, to connect this back to unconditional loops:** what happens if you write `while True: ... else: ...`? The condition is the literal `True` — it can *never* become falsy on its own, so the loop can only ever end via `break` (or `return`/`raise`). That means the `else` block is **unreachable dead code**. Seeing `while True:` paired with an `else` is a signal that the tool doesn't match the loop shape: reach for a real, checkable condition (like `while attempts_left:` below) whenever you actually want an `else`.

In [ ]:
import random

random.seed(4)  # deterministic output for the tutorial


def flaky_call() -> bool:
    """Pretend API call that succeeds ~40% of the time."""
    return random.random() < 0.4


attempts_left = 3  # a real, checkable condition -> while-else is meaningful here

while attempts_left:  # conditional loop: bounded by attempts_left, not "while True"
    attempts_left -= 1
    if flaky_call():
        print("call succeeded")
        break  # early success -> skip the else entirely
else:
    # only reached if we ran out of attempts_left WITHOUT ever succeeding
    print("exhausted retries — giving up")


### 4. Internals — what's actually happening under the hood

There is no special "else machinery" at the bytecode level. The `else` block is simply the statement that comes right after the loop in the compiled code — the same place execution would land if there were no `else` at all. `break` compiles to a jump instruction that lands *past* that block; a normal loop exit (condition false / iterator exhausted) just falls through *into* it. That is the whole mechanism — which is exactly why the flag-variable rewrite above behaves identically: `else` is syntactic sugar for "skip this block if a break happened," and the flag was you implementing that skip by hand.

In [ ]:
import dis


def search(items, target):
    for item in items:
        if item == target:
            break
    else:
        print("not found")


dis.dis(search)
# Look for: no opcode named "ELSE" anywhere. break's JUMP target lands right where the
# "not found" bytecode already lives — the block you wrote after `else:` is just ordinary
# code that sits after the loop, reached by falling through when no break jump happened.


### 5. Gotchas, decision rules, and the interview angle

**More gotchas specific to loop-`else`:**
- `continue` does **not** skip the `else` — only `break` does. `continue` just ends the current iteration and goes back to the loop; if the loop then exits normally, `else` still runs.
- If the loop body raises an uncaught exception, execution leaves the loop *and* skips `else` — but not because of some `else`-specific rule; the exception is propagating past the whole statement, `else` included. `else` is not a `finally` — it does not run "no matter what."
- `else` does **not** let you break out of a nested (outer) loop. `break` only escapes its own immediate loop, so wrapping that in `for-else` doesn't reach further out either. For "stop two loops at once" you need a flag checked in both loops, an exception, or (cleanest) extracting the inner loop into a function and using `return`.

**Decision rules (use these to pick the tool, not vibes):**

| Situation | Reach for |
|---|---|
| Need to react to "loop ended without success" | `for/while ... else` — skip the flag variable |
| Need to react to "loop ended without success" *and* also stop an outer loop | a flag checked by both loops, or extract to a function + `return`/`raise` |
| Just need a yes/no "does any/all item satisfy X" | `any()` / `all()` — no loop needed at all |
| Need the *matching item itself*, not just yes/no | `for-else` (or `next((x for x in it if cond), default)`) |
| Need to abandon the loop the instant a goal is met | `break` |
| Need to skip just one bad/irrelevant item, keep the rest going | `continue` (or, if it's pure filtering with no side effects, a comprehension) |
| Loop should run until an internally-detected event, with no natural end | `while True:` + `break` |
| Loop is bounded by a real, checkable condition | `while condition:` — and now `else` is meaningful if you need it |

**Interview angle:** an interviewer bringing up `for-else` is usually testing two things at once — (1) do you actually know the rule (skip on `break`, not on "loop ran"/"found nothing to iterate"), and (2) do you have the judgment to know it's a niche, divisive tool (many style guides ban it for readability) that you'd use sparingly, not by default. A strong answer states the rule precisely, shows the flag-variable equivalent unprompted, and volunteers that `try/except/else` follows the identical "no early/abnormal exit" rule.

---

## Practice Problems

Goal: build instant recognition of *which tool fits which shape of problem* — especially `for/while-else`, `break`, and `continue`, since those are the ones that don't click until you've hit their canonical use case a few times.

For every problem, before you write code, answer in a comment: **"what would this look like without the tool, and why is that worse?"** That's the question that made the tutorial sections above make sense — apply it here yourself.

Problems are grouped by the pattern they drill. Solve in order the first time through; come back and redo just one group later for spaced practice.

### Group A — `for-else`: the search / validate pattern

#### Problem 1 — Prime checker (the textbook `for-else` use case)

Write `is_prime(n: int) -> bool`. For `n >= 2`, loop candidate divisors from `2` up to `sqrt(n)`; `break` the moment you find one that divides `n` evenly. Use `for-else` to decide the return value — no `found`/`is_divisible` flag variable allowed. This is the single most-cited canonical example of why `for-else` exists: "checked every candidate divisor without success" *is* the definition of prime.

In [ ]:
def is_prime(n: int) -> bool:
    # TODO: for-else over candidate divisors, no flag variable
    ...


for n in (1, 2, 17, 18, 97, 100):
    print(n, is_prime(n))


#### Problem 2 — First matching log line, with a proper "not found" path

You have a list of log lines (strings). Write `find_first_error(lines: list[str]) -> str`, which scans for the first line containing the substring `"ERROR"` and returns it immediately when found. If no line matches, an `else` clause on the loop should raise `LookupError("no error line found")`. (Notice `return` alone gets you out of the whole function without ever needing an explicit `break` — and it skips `else` for the same reason `break` does: the loop never reached its natural end.) Then write a second version, `find_first_error_flagged`, using an explicit flag variable instead — same behavior, different mechanism — so you can compare them side by side.

In [ ]:
log_lines = [
    "12:00 INFO service started",
    "12:01 INFO health check ok",
    "12:03 ERROR db connection timeout",
    "12:04 INFO retrying",
]

log_lines_clean = ["12:00 INFO ok", "12:01 INFO ok"]


def find_first_error(lines: list[str]) -> str:
    # TODO: for-else, return inside the loop, raise LookupError in else
    ...


def find_first_error_flagged(lines: list[str]) -> str:
    # TODO: same behavior, explicit flag variable instead of for-else
    ...


for finder in (find_first_error, find_first_error_flagged):
    print(finder.__name__, "->", finder(log_lines))
    try:
        finder(log_lines_clean)
    except LookupError as exc:
        print(finder.__name__, "->", exc)


#### Problem 3 — Batch validation: `for-else` vs. `all()`

Write `validate_batch(orders: list[dict]) -> None`, where each order is `{"id": int, "total": float}`. It should loop through, and if it finds an order with `total < 0`, `break` and print `f"invalid order {id}"`. Use `for-else` so the `else` prints `"all orders valid"` only if every single one passed.

Then write a one-line alternative using `all(order["total"] >= 0 for order in orders)`.

**The decision rule to articulate for yourself afterward:** when do you reach for `for-else` instead of `all()`/`any()`? (Hint: what does `for-else` give you access to inside the loop that a bare boolean check doesn't?)

In [ ]:
orders = [{"id": 1, "total": 10.0}, {"id": 2, "total": -5.0}, {"id": 3, "total": 3.0}]
clean_orders = [{"id": 1, "total": 10.0}, {"id": 2, "total": 3.0}]


def validate_batch(orders: list[dict]) -> None:
    # TODO: for-else, break + print on the first invalid order, print success in else
    ...


validate_batch(orders)
validate_batch(clean_orders)

# TODO: one-line all() equivalent


### Group B — `break` / `continue` fundamentals

#### Problem 4 — Skip bad sensor readings, keep the batch alive

Write `average_valid_readings(readings: list[str]) -> float`. Each reading is a string that should parse as a float between `-50` and `60` (a plausible Celsius range). Use `continue` to skip any reading that isn't a valid float, or is out of range — do **not** let one bad reading crash the whole computation or get excluded via a pre-filtered comprehension (the point here is practicing `continue` with multiple independent skip-conditions in the loop body, not writing the shortest possible one-liner).

In [ ]:
readings = ["21.5", "not-a-number", "19.0", "999", "-60", "22.1", ""]


def average_valid_readings(readings: list[str]) -> float:
    # TODO: continue past unparsable / out-of-range readings, average the rest
    ...


print(average_valid_readings(readings))


#### Problem 5 — Stop pagination once you have enough results

You're given `fetch_page(page_number: int) -> list[dict]`, a simulated API call (provided below, deterministic). Write `collect_results(target_count: int) -> list[dict]` that calls `fetch_page` for `page_number = 1, 2, 3, ...`, extending an accumulator list, and `break`s the instant the accumulator has **at least** `target_count` items — even if that means using only part of the last page. Also guard against a runaway loop: if a page comes back empty, treat that as "no more data" and `break` too. This is the canonical "stop once the goal is met" shape — the loop has no natural end of its own (you don't know how many pages exist ahead of time).

In [ ]:
PAGE_SIZE = 10
TOTAL_ITEMS = 25


def fetch_page(page_number: int) -> list[dict]:
    """Given: simulated paginated API, page_number starts at 1, empty list = no more data."""
    start = (page_number - 1) * PAGE_SIZE
    end = min(start + PAGE_SIZE, TOTAL_ITEMS)
    if start >= TOTAL_ITEMS:
        return []
    return [{"id": i} for i in range(start, end)]


def collect_results(target_count: int) -> list[dict]:
    # TODO: while True (unconditional -- unknown page count) + break on target reached
    #       or empty page, using an unconditional loop since there's no natural end
    ...


results = collect_results(15)
print(len(results), results)


#### Problem 6 — Find and fix the infinite loop

The snippet below is **buggy — do not run it as-is**, it will hang forever. It's meant to print only the even numbers from `0` to `9`:

```python
i = 0
while i < 10:
    if i % 2 != 0:
        continue  # skip odd numbers
    print(i)
    i += 1
```

In the code cell below, write a comment explaining *exactly* why this hangs (name the mechanism from the gotchas section, don't just say "it's broken"), then write the corrected version and run it to confirm it prints `0 2 4 6 8` and terminates.

In [ ]:
# TODO: comment explaining why the snippet above hangs, then the corrected loop


### Group C — `while-else` and conditional-vs-unconditional loop patterns

#### Problem 7 — Login attempt lockout (`while-else`, conditional loop)

Write `attempt_login(correct_password: str, attempts: list[str], max_attempts: int = 3) -> bool`. Use a **conditional** `while` loop bounded by a remaining-attempts counter, checking one guess from `attempts` per iteration. `break` the moment a guess matches `correct_password` and return `True`. If the counter runs out without a match, use the loop's `else` to print `"account locked"` and return `False`. This is the standard shape for "bounded number of tries, distinct success vs. exhaustion outcomes" — the same shape as a retry-with-backoff against a flaky service.

In [ ]:
def attempt_login(correct_password: str, attempts: list[str], max_attempts: int = 3) -> bool:
    # TODO: conditional while (bounded by remaining attempts) + break on match + while-else lockout
    ...


print(attempt_login("s3cret", ["wrong1", "s3cret", "wrong2"]))
print(attempt_login("s3cret", ["wrong1", "wrong2", "wrong3", "wrong4"]))


#### Problem 8 — Tiny calculator REPL (`while True`, unconditional loop)

Write `run_calculator(commands: list[str]) -> list[float]`. Each command is either `"add 2 3"`, `"sub 10 4"`, `"mul 3 3"`, or `"exit"`. Loop with `while True:` (there's no natural stopping condition — you only know to stop when you see `"exit"`), computing and collecting a result for each arithmetic command, and `break` when you hit `"exit"`. Feed it the `commands` list below via an iterator (like the sentinel-loop tutorial example) instead of real `input()`, so it stays runnable and testable. Notice there's no `else` here at all — explain in a comment why an `else` on this loop would be exactly as dead as the `while True: ... else:` trap from the tutorial.

In [ ]:
commands = ["add 2 3", "mul 3 3", "sub 10 4", "exit", "add 99 1"]  # "exit" should stop it early


def run_calculator(commands: list[str]) -> list[float]:
    # TODO: while True + break on "exit"; comment on why `else` here would be dead code
    ...


print(run_calculator(commands))


#### Problem 9 — Find a resource across paginated results, or raise a domain error

Define `class ResourceNotFoundError(Exception): ...`. Using the same `fetch_page` from Problem 5, write `find_resource(resource_id: int) -> dict`, which walks pages (unconditional `while True`, since you don't know the page count up front) looking for `{"id": resource_id}`. `break` the outer search the instant it's found (you'll need a nested `for` over each page's items *inside* the `while True` — this is a preview of the nested-loop problem next: notice you cannot simply `break` the inner `for` and expect the outer `while` to stop too). If you exhaust all pages (an empty page comes back) without a match, raise `ResourceNotFoundError(resource_id)`. This mirrors scanning paginated API responses for one record — the same shape as a `for-else`, just wrapped in an outer unconditional loop because the page count isn't known ahead of time.

In [ ]:
class ResourceNotFoundError(Exception):
    ...


def find_resource(resource_id: int) -> dict:
    # TODO: outer while True over pages; inner for-else over one page's items;
    #       raise ResourceNotFoundError if every page is exhausted without a match
    ...


print(find_resource(17))
try:
    find_resource(999)
except ResourceNotFoundError as exc:
    print(f"not found: {exc}")


### Group D — nested loops and pulling it all together

#### Problem 10 — Find a target in a 2D grid, stopping *both* loops

Write `find_in_grid(grid: list[list[int]], target: int) -> tuple[int, int] | None`, returning `(row, col)` of the first occurrence of `target`, or `None`. Deliberately first write a version where you `break` the inner `for` when found — and observe that the outer `for` just keeps going, scanning rows you no longer care about (add a `print` to prove it). Then fix it using **one** of: (a) a `found` flag checked at the top of the outer loop, or (b) extracting the whole search into a function so you can `return` directly from inside the inner loop. This is the direct, hands-on version of the "`break`/`else` don't reach outer loops" gotcha.

In [ ]:
grid = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
]


def find_in_grid_broken(grid: list[list[int]], target: int):
    # TODO: inner break only -- print each row visited to PROVE the outer loop doesn't stop
    ...


def find_in_grid(grid: list[list[int]], target: int) -> tuple[int, int] | None:
    # TODO: fixed version -- flag checked by outer loop, OR return from a nested function
    ...


find_in_grid_broken(grid, 5)
print(find_in_grid(grid, 5))
print(find_in_grid(grid, 42))


#### Problem 11 — Same rule, two loop shapes: `for-else` and `while-else` side by side

Write `all_usernames_valid_for(usernames: list[str], max_length: int) -> bool` using a `for-else` over the list: `break` and print `f"'{name}' too long"` on the first username longer than `max_length`; `else` prints `"all usernames valid"` and the function returns accordingly (`True`/`False`).

Now write `all_usernames_valid_while(usernames: list[str], max_length: int) -> bool` with **identical** behavior, but built on a `while` loop with a manual index counter as the condition, instead of iterating the list directly. Run both on the same inputs and confirm the outputs match exactly. The point: `for-else` and `while-else` are the *same rule* ("did a break happen?") wearing two different loop shapes — internalizing that they're interchangeable is what makes the choice between them a style question, not a correctness one.

In [ ]:
usernames_ok = ["alice", "bob", "carol"]
usernames_bad = ["alice", "a_very_long_username_here", "carol"]


def all_usernames_valid_for(usernames: list[str], max_length: int) -> bool:
    # TODO: for-else version
    ...


def all_usernames_valid_while(usernames: list[str], max_length: int) -> bool:
    # TODO: while-else version, manual index counter as the loop condition
    ...


for validator in (all_usernames_valid_for, all_usernames_valid_while):
    print(validator.__name__, validator(usernames_ok, 10), validator(usernames_bad, 10))


#### Problem 12 — Diagnostic: does `else` fire? Predict first, then check

This is the "instant recognition" drill. For **each** snippet below, write down your prediction — will `"else ran"` print or not — and *why*, before running anything. Then uncomment and run each one individually (they're kept separate so one doesn't short-circuit the next) to check yourself. Getting every one right on the first guess is the actual goal of this whole notebook.

```python
# (a)
for x in [1, 2, 3]:
    pass
else:
    print("else ran")

# (b)
for x in [1, 2, 3]:
    if x == 2:
        break
else:
    print("else ran")

# (c)
for x in [1, 2, 3]:
    if x == 99:
        break
else:
    print("else ran")

# (d)
for x in []:
    break
else:
    print("else ran")

# (e)
for x in [1, 2, 3]:
    if x == 2:
        continue
else:
    print("else ran")

# (f)
for x in [1, 2, 3]:
    if x == 2:
        raise ValueError("boom")
else:
    print("else ran")
```

In [ ]:
# Predictions (fill in "yes"/"no" + one-clause reason BEFORE running anything below):
# (a) ...
# (b) ...
# (c) ...
# (d) ...
# (e) ...
# (f) ...

# Then copy each snippet from the markdown cell above down here one at a time to verify.
